In [10]:
# IMPORT LIBRARIES
import os
import subprocess
from typing import Dict, Any, Callable

from dotenv import load_dotenv
import json
import requests

load_dotenv('./env', override=True)

CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')

# GET TOKEN
## POST request to obtain access token
def get_token(client_id: str, client_secret: str, url: str) -> Dict[Any, Any]:
  """Perform a POST request to obtain an access token.

  Args:
    client_id (str): App client id
    client_secret (str): App client secret
    url (str): URL to perform the post request

  Returns:
    Dict[Any, Any]: Dictionary containing the access toke
  """

  headers = {
      "Content-Type": "application/x-www-form-urlencoded"
  }

  payload = {
             "grant_type": "client_credentials",
             "client_id": client_id,
             "client_secret": client_secret
            }

  try:
    response = requests.post(url=url, headers=headers, data=payload)
    print(type(response))
    response.raise_for_status()
    response_json = json.loads(response.content)

    return response_json

  except Exception as err:
    print(f"Error: {err}")
    return{}

URL_TOKEN="https://accounts.spotify.com/api/token"
token = get_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET, url=URL_TOKEN)

print(token)

## Use acess token to return authorization header
def get_auth_header(access_token: str) -> Dict[str, str]:
  return {"Authorization": f"Bearer {access_token}"}

# GET FEATURED PLAYLISTS
def get_featured_playlists(url: str, access_token: str, offset: int=0, limit: int=20, next: str="") -> Dict[Any, Any]:
  """Perform get() request ot featured playlists endpoint

  Args:
    url (str): Base url for the request
    access_token (str): Access token
    offset (int, optional): Page offset for pagination. Defaults to 0.
    limit (int, optional): Number of elements per page. Defaults to 20.
    next (str, optional): Next URL to perform next request. Defaults to "".

  Returns:
    Dict[Any, Any]: Request response
  """

  if next == "":
    request_url = f"{url}?offset={offset}&limit={limit}"
  else:
    request_url = f"{url}"

  # Call get_auth_header() function and pass the access token.
  # Changed keyword argument from 'access_token' to 'token' to match function definition.
  headers = get_auth_header(access_token=access_token)

  try:
    # Peform a get() request using the request_url and headers.
    print(f"Request URL: {request_url}")
    response = requests.get(url=request_url, headers=headers)
    # Check if the request was successful
    response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx)
    # Use json() method over the response to return it as a Python dictionary.
    return response.json()

  except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        return {'error': {'status': response.status_code, 'message': response.reason}}
  except Exception as err:
        print(f"Other error occurred: {err}")
        return {'error': {'status': 'unknown', 'message': str(err)}}


URL_FEATURE_PLAYLISTS = "https://api.spotify.com/v1/browse/featured-playlists"

playlists_response = get_featured_playlists(url=URL_FEATURE_PLAYLISTS, access_token=token.get('access_token'))
playlists_response

## Sturcture of the response
playlists_response.keys()
playlists_response.get('playlists').keys()

# URL for request sent
playlists_response.get('playlists').get('href')

# Returned items
len(playlists_response.get('playlists').get('items'))

<class 'requests.models.Response'>
{'access_token': 'BQDkI4ilUNr-b4oz8E4BjU_MuJzxW9ycoGu-fYGvMKayBNGBORx4LixzT37F5RICBE-E1nPvcaPM0IGo7_BFwjhDonh5KYvXna7rGTGtKY421_WtY_szm98qdbO9AmZZHIx2ABEHEPQ', 'token_type': 'Bearer', 'expires_in': 3600}
Request URL: https://api.spotify.com/v1/browse/featured-playlists?offset=0&limit=20
HTTP error occurred: 404 Client Error: Not Found for url: https://api.spotify.com/v1/browse/featured-playlists?offset=0&limit=20


AttributeError: 'NoneType' object has no attribute 'keys'